In [2]:
# pip install faiss-cpu

In [3]:
import spacy
import faiss
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
nlp = spacy.load('en_core_web_sm')

In [5]:
data = open('machine_learning_2000_sentences.txt', encoding='utf-8').read()

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 100, chunk_overlap = 20)

In [7]:
chunks = splitter.split_text(data)

In [8]:
# print(chunks, type(chunks), type(chunks[0]))

In [9]:
embedding_model = SentenceTransformer(
    model_name_or_path='sentence-transformers/all-miniLM-L6-V2'
)

'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-miniLM-L6-V2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

why we convert to vectors using embedding
- to compare values with vector database stored values

In [ ]:
embeddings = embedding_model.encode(chunks).astype('float32')

In [ ]:
embeddings

array([[-0.07728466, -0.00678861,  0.05914882, ...,  0.06669836,
        -0.07794894,  0.01822445],
       [ 0.01255796, -0.04760394,  0.02898123, ...,  0.04792494,
        -0.02147599, -0.02757207],
       [ 0.0154788 , -0.00400942, -0.09803507, ...,  0.04105116,
         0.03302541, -0.01764336],
       ...,
       [-0.01283604, -0.00381554,  0.05603732, ...,  0.08069879,
         0.06248713, -0.05116992],
       [-0.03425974, -0.02970343, -0.01779654, ...,  0.0584114 ,
         0.03584176, -0.00811812],
       [ 0.02394698,  0.00384395,  0.05797815, ..., -0.13486075,
         0.00705718,  0.03490416]], shape=(7240, 384), dtype=float32)

In [ ]:
dimension = embeddings.shape[1]
dimension

384

In [ ]:
faiss

<module 'faiss' from 'c:\\Users\\LENOVO\\AppData\\Local\\Programs\\Python\\Python314\\Lib\\site-packages\\faiss\\__init__.py'>

In [ ]:
# in rag, first step is normalize the values between 0 and 1 (convert all embeddings to same scale)
faiss.normalize_L2(embeddings)  # returns None means it stores the values into faiss

it will only accept array float32
- internally it uses numpy

In [ ]:
# (cosine similarty and euclidean distance)
index = faiss.IndexFlatIP(dimension) # it uses dot product 
# index = faiss.IndexFlatL2() # euclidean distance

In [ ]:
index.add(embeddings) # stored embeddings inside faiss DB

SEARCH

In [ ]:
query = "Explain Machine Learning?"

In [ ]:
query_embedding = embedding_model.encode(query).astype('float32')

In [ ]:
query_embedding.shape

(384,)

In [ ]:
# [query_embedding]

In [ ]:
query_embedding.reshape(1,-1).shape

(1, 384)

In [ ]:
query_embedding = query_embedding.reshape(1,-1)

In [ ]:
faiss.normalize_L2(query_embedding) # shape mismatch

SEARCH METHOD

# very important interview question
what is k?

TypeError: handle_Index.<locals>.replacement_search() missing 1 required positional argument: 'k'

In [ ]:
index.search(query_embedding, k = 3) # it will give error because it is text format
# returns 2 values
# 1st value = distance
# 2nd value -> index value -> which row the chunk is present
# k = 2 -> returns top 2 similar chunks
# embeddings create 7200 chunks

(array([[0.619247  , 0.6190823 , 0.61763775]], dtype=float32),
 array([[3189, 6726, 6574]]))

In [ ]:
index.search(query_embedding, k = 5) 
# k value is not fixed, we need to check manually which k value is suitable

(array([[0.619247  , 0.6190823 , 0.61763775, 0.61221784, 0.6106998 ]],
       dtype=float32),
 array([[3189, 6726, 6574, 3330, 1379]]))

# to know whether the retriveal is correct or not(distance) we need to perform Retrieval Testing

TODAY'S TASK
create udf, def rag_query(query) -> return two values, distance and index

- search -> distance and index
- search documents -> retrieval chunks

In [ ]:
index.search

GOOGLE LLM 30 july


In [ ]:
# task
def rag_query(query):
    # Convert the query into an embedding
    query_embedding = embedding_model.encode(query).astype("float32")

    # Reshape into a 2D array
    query_embedding = query_embedding.reshape(1, -1)

    # Normalize the query embedding
    faiss.normalize_L2(query_embedding)

    # Perform semantic search
    distances, indices = index.search(query_embedding, k=3)

    # Return the similarity scores and indices
    return distances, indices

query = 'Describe in brief, the types of Machine Learning.'
rag_query(query)